<a href="https://colab.research.google.com/github/shivaprajapati34390-netizen/ML-project/blob/main/Predictive_keyboard_model_using_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


step 1 Prepareing the datasets

In [1]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab') # Added to resolve LookupError

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [23]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

# load data
with open('/content/sample_data/shivasher.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

tokens = word_tokenize(text)
print("Total Tokens:", len(tokens))

Total Tokens: 1106


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



step 2 creating the vocabulary

In [24]:
from collections import Counter

In [25]:
word_counts=Counter(tokens)
vocab=sorted(word_counts,key=word_counts.get,reverse=True)


In [38]:
word2idx={word: idx for idx,word in enumerate(vocab)}
idx2word={idx: word for idx,word in enumerate(vocab)}
vocab_size=(len(vocab))

step 3 building input output sequences

In [27]:
from typing import Sequence
Sequence_length=4 # e.g., "I am going to [predict this]"



In [28]:
data=[]
for i in range (len(tokens)-Sequence_length):
  input_seq=tokens[i:i+Sequence_length-1]
  target_word=tokens[i+ Sequence_length-1]
  data.append((input_seq,target_word))



In [29]:
import torch
# convert word to indices
def encode(seq): return [word2idx[word] for word in seq]

encoded_data=[(torch.tensor(encode(inp)),torch.tensor(word2idx[target])) for inp,target in data]


step 4 Designing the model architecture

In [30]:
import torch.nn as nn


In [31]:
class PredictKeyboard(nn.Module):
  def __init__(self,vocab_size,embed_dim=64,hidden_dim=128):
    super(PredictKeyboard,self).__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
    self.fc=nn.Linear(hidden_dim,vocab_size)

  def forward(self,x):
    x=self.embedding(x)
    output, _ = self.lstm(x) # Correctly unpack output and discard hidden/cell states
    output=self.fc(output[:,-1,:]) #last lstm output
    return output

step-5 traning the model

In [32]:
import torch
import torch.optim as optim
import random

In [33]:
model=PredictKeyboard(vocab_size)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.005)


In [34]:
epochs=20
for epoch in range(epochs):
  total_loss=0
  random.shuffle(encoded_data)
  for input_seq,target in encoded_data[:10000]: #limit data for speed
   input_seq=input_seq.unsqueeze(0)
   output=model(input_seq)
   loss=criterion(output,target.unsqueeze(0))
   optimizer.zero_grad()
   loss.backward()
   optimizer.step()
   total_loss+=loss.item()
  print(f"epoch {epoch+1}.loss:{total_loss:.4f}")

epoch 1.loss:5176.2003
epoch 2.loss:3423.9647
epoch 3.loss:1917.0501
epoch 4.loss:1060.7972
epoch 5.loss:635.9610
epoch 6.loss:452.9122
epoch 7.loss:375.2924
epoch 8.loss:374.9315
epoch 9.loss:409.1058
epoch 10.loss:579.8102
epoch 11.loss:521.4512
epoch 12.loss:372.2313
epoch 13.loss:316.3522
epoch 14.loss:291.1910
epoch 15.loss:298.7612
epoch 16.loss:277.1652
epoch 17.loss:293.4294
epoch 18.loss:536.2233
epoch 19.loss:842.0406
epoch 20.loss:362.7065


In [35]:
print(f"epoch {epoch+1}.loss:{total_loss:.4f}")

epoch 20.loss:362.7065


In [36]:
import torch.nn.functional as F

In [44]:
def suggest_next_words(model,text_prompt,top_k=3):
  model.eval()
  tokens=word_tokenize(text_prompt.lower())
  if len(tokens)< Sequence_length -1:
    raise ValueError(f"Input should be at least {Sequence_length -1} words longs")
  input_seq=tokens[-(Sequence_length -1):]
  input_tensor=torch.tensor(encode(input_seq)).unsqueeze(0)

  with torch.no_grad():
    output=model(input_tensor)
    probs=F.softmax(output, dim=1).squeeze()
    top_indicies=probs.topk(top_k).indices.tolist() # Corrected to use top_k as first argument
    print(f"DEBUG: top_indicies = {top_indicies}")
    print(f"DEBUG: idx2word = {idx2word}") # Print the actual dict at runtime
    return[idx2word[idx]for idx in top_indicies]
# Example usage with words from the current vocabulary
print("suggestion",suggest_next_words(model," you are a "))

DEBUG: top_indicies = [188, 231, 260]
DEBUG: idx2word = {0: '*', 1: '(', 2: ')', 3: 'to', 4: ',', 5: '-', 6: 'a', 7: 'the', 8: 'is', 9: ';', 10: 'in', 11: '>', 12: 'backend', 13: '[', 14: ']', 15: '``', 16: 'use', 17: ':', 18: 'and', 19: "''", 20: 'frontend', 21: 'file', 22: '.', 23: 'server', 24: '{', 25: '}', 26: '=', 27: "'", 28: 'that', 29: 'cors', 30: 'express', 31: 'or', 32: 'your', 33: 'import', 34: 'we', 35: 'data', 36: 'used', 37: 'it', 38: '--', 39: 'main', 40: 'install', 41: 'req', 42: 'res', 43: 'next', 44: 'run', 45: 'api', 46: 'app.use', 47: 'proxy', 48: 'code', 49: 'const', 50: 'pass', 51: 'mai', 52: 'package.json', 53: 'npm', 54: 'module', 55: 'require', 56: 'start', 57: 'running', 58: 'like', 59: 'requests', 60: 'request', 61: '//', 62: 'client', 63: '3.', 64: 'see', 65: 'for', 66: 'common.js', 67: '&', 68: 'not', 69: 'when', 70: 'fetch', 71: 'our', 72: 'from', 73: 'page', 74: 'http', 75: 'if', 76: 'res.send', 77: 'thunder', 78: 'get', 79: 'post', 80: 'how', 81: 'setup